# Ollama + Color — direct image processing (no IIIF)
# 
# Downloads each image directly from the URL, calls parsers locally.
# No Flask server needed. Works with any direct image URL (CloudFront, etc.).

In [5]:
import sys, os, tempfile, time, csv, random
import requests
import pandas as pd
from IPython.display import Image, display

# parsers/ is the cwd when running from here
if "." not in sys.path:
	sys.path.insert(0, ".")

from ollama import OllamaModel, Ollama
from colors import Colors

In [6]:
IMAGES_CSV = "../temp/images.csv"
OUT_DIR    = "../results"
os.makedirs(OUT_DIR, exist_ok=True)

images = pd.read_csv(IMAGES_CSV)
print(f"{len(images)} images loaded")
images.head(3)

2053 images loaded


,Title,Identifier,Artist,Date,Image URL,Alt Text,Is Primary
0,Robert Hamilton Bishop (1777-1855),1829.P.1.1,Horace Harding,1829-1830,https://d1y502jg6fpugt.cloudfront.net/29278/ar...,Robert Hamilton Bishop (1777-1855),1
1,"Portrait of John Williamson Herron, (1827-1912)",1905.P.1.1,"Annette Covington, American 1872-1964",1845,https://d1y502jg6fpugt.cloudfront.net/29278/ar...,"Portrait of John Williamson Herron, (1827-1912)",1
2,Group of Farm Animals with Horse and Rider,1909.P.1.1,James H. Beard,NaN,https://d1y502jg6fpugt.cloudfront.net/29278/ar...,Group of Farm Animals with Horse and Rider,1


In [7]:
import re
from urllib.parse import urlparse

TEMP_DIR = "../temp"

def local_cache_path(url):
	"""Return the path where cache.py would store the full-size image for this URL."""
	parsed = urlparse(url)
	domain = parsed.netloc
	last_segment = [s for s in parsed.path.split("/") if s][-1] if parsed.path else "image"
	basename = re.sub(r"[^a-zA-Z0-9._-]", "_", last_segment)
	return os.path.join(TEMP_DIR, "full", domain, basename + ".jpg")

def download(url):
	"""Return a local path for the image — uses temp cache if already downloaded."""
	cached = local_cache_path(url)
	if os.path.exists(cached):
		return cached, False  # (path, is_temp)

	resp = requests.get(url, timeout=30)
	resp.raise_for_status()
	suffix = ".png" if "png" in url.lower() else ".jpg"
	tmp = tempfile.NamedTemporaryFile(delete=False, suffix=suffix)
	tmp.write(resp.content)
	tmp.close()
	return tmp.name, True  # (path, is_temp)

def run(image_url, model=OllamaModel.GEMMA_4, include_colors=True):
	"""Download image (or use cache) and run Ollama + color parser."""
	path, is_temp = download(image_url)
	try:
		result = Ollama().fetch(path, model)
		if include_colors:
			result["colors"] = Colors().fetch(path)
		return result
	finally:
		if is_temp:
			os.unlink(path)

In [8]:
# Quick test — one image
test_url = images["Image URL"].iloc[0]
display(Image(url=test_url))

result = run(test_url)
print("status :", result["status"])
print("model  :", result.get("model"))
print("colors :", len(result.get("colors", [])), "colors extracted")
print()
print(result.get("body", "")[:500])

status : 200
model  : gemma4:latest
colors : 4 colors extracted

Based on the image provided, here is a detailed description:

**Overview:**
This is a formal, half-length portrait painting of a man, executed in a style typical of 18th or early 19th-century academic art. The mood is somber and intellectual, giving the subject an air of gravitas and importance.

**The Subject:**
The man depicted is seated and gazes directly out at the viewer with a serious, steady expression. He has dark hair styled relatively formally for the period. His posture is upright and


# Benchmark — 100 random images, both models

In [9]:
SAMPLE_SIZE   = 5
SEED          = 1809
BENCHMARK_CSV = f"{OUT_DIR}/benchmark.csv"
MODELS        = [OllamaModel.GEMMA_4, OllamaModel.GEMMA_4_26B]

# Same 100 images for both models
sample = images.sample(SAMPLE_SIZE, random_state=SEED).reset_index(drop=True)
print(f"Sample locked to seed={SEED}: {len(sample)} images")

Sample locked to seed=1809: 5 images


In [ ]:
BENCHMARK_CSV = f"{OUT_DIR}/benchmark.csv"
MODELS        = [OllamaModel.GEMMA_4, OllamaModel.GEMMA_4_26B]

rows = []

for model in MODELS:
	print(f"\n### {model.name}  ({model.model_id})")
	for i, row in sample.iterrows():
		image_url = row["Image URL"]
		t0 = time.perf_counter()
		result = run(image_url, model=model, include_colors=False)
		elapsed = time.perf_counter() - t0

		body = result.get("body") or ""
		cold = " (cold start)" if i == 0 else ""
		flag = "" if result["status"] == 200 else "  <-- ERROR"
		print(f"  [{i+1:3}] {row['Identifier']:<20} {elapsed:6.1f}s  {len(body):5d} chars{cold}{flag}")

		rows.append({
			"model":       model.name,
			"identifier":  row["Identifier"],
			"title":       row["Title"],
			"status":      result["status"],
			"runtime_s":   round(elapsed, 1),
			"chars":       len(body),
			"description": body,
		})

	# Save after each model so progress isn't lost if interrupted
	pd.DataFrame(rows).to_csv(BENCHMARK_CSV, index=False)

df_results = pd.DataFrame(rows)
print(f"\nDone — {len(df_results)} rows saved to {BENCHMARK_CSV}")
df_results[["model", "identifier", "status", "runtime_s", "chars"]]


### gemma4  (gemma4:latest)
  [  1] obj-02155              21.8s   1403 chars (cold start)
  [  2] 1993.46                39.3s   2807 chars
  [  3] 2024.11                33.5s   2056 chars
  [  4] 2006.321               37.2s   2453 chars
